# Packages

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datetime import datetime, timedelta

import random

# Data Import

## auction price

In [3]:
offset = (
    pd.Timestamp.today().normalize()
    - pd.Timestamp("2026-06-01")
).days

today = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize()

today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

FTRAuctions = pd.read_csv(f'data/FTRAuction/FTRAuctions_{today_str}.csv', skiprows=[1])

FTRAuctions.rename(columns={"EndDate": "AuctionEndDate"}, inplace=True)

FTRAuctions.head()

,AuctionKey,ISOCode,AuctionName,StartDate,AuctionEndDate,AuctionRound,AuctionDate,AuctionStatusCode,ResultsPostedDate,ISOAuctionName
0,3205,PJM,Long Term 2026-29 RD 5,2026-03-03 00:00:00.000,2026-03-05 00:00:00.000,5,2026-06-01 00:00:00.000,CLOSED,2026-03-11 00:00:00.000,26/29 Long Term Auction
1,3210,PJM,Long Term 2026-29 RD 5 Credit Study,2026-03-03 00:00:00.000,2026-03-05 00:00:00.000,5,2026-06-01 00:00:00.000,CLOSED,2026-03-11 00:00:00.000,26/29 Long Term Auction Credit Study
2,3219,PJM,FEB 2026 Auction,2026-01-16 00:00:00.000,2026-01-21 00:00:00.000,1,2026-02-01 00:00:00.000,CLOSED,2026-01-28 00:00:00.000,FEB 2026 Auction
3,3220,PJM,MAR 2026 Auction,2026-02-11 00:00:00.000,2026-02-13 00:00:00.000,1,2026-03-01 00:00:00.000,CLOSED,2026-02-21 00:00:00.000,MAR 2026 Auction
4,3221,PJM,APR 2026 Auction,2026-03-12 00:00:00.000,2026-03-16 00:00:00.000,1,2026-04-01 00:00:00.000,OPEN,2026-03-23 00:00:00.000,APR 2026 Auction


In [4]:
FTRAPeriods = pd.read_csv(f'data/FTRAuction/FTRAuctionPeriods_{today_str}.csv', skiprows=[1])

FTRAPeriods.head()

,AuctionKey,PeriodCode,StartDate,EndDate,PeakHours,OffpeakHours,H24Hours,PeriodTypeCode,ISOCode,DailyOffPeakHours,WkndOnPeakHours,ISOPeriodCode,PNodeBidType
0,3117,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
1,3118,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
2,3119,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
3,3120,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
4,3121,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM


In [5]:
NodePrices = pd.read_csv(f'data/FTRAuction/FTRAuctionNodePrices_{today_str}.csv', skiprows=[1])

NodePrices.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,CongOnPeakMwh,CongOffPeakMwh,ISOCode,CongDailyOffPeakMwh,CongWkndOnPeakMwh,UploadDate
0,3117,PJM-YR-24-25,3,-1.875437,-2.660866,-1.195713,PJM,-0.847000,-1.769049,2023-06-12 16:46:30.857
1,3117,PJM-YR-24-25,4,4.054796,5.666407,2.660079,PJM,1.993459,3.756098,2023-06-12 16:46:30.857
2,3117,PJM-YR-24-25,5,4.495161,5.970529,3.218352,PJM,2.562705,4.296329,2023-06-12 16:46:30.857
3,3117,PJM-YR-24-25,6,10.011016,13.348408,7.122779,PJM,5.465387,9.847771,2023-06-12 16:46:30.857
4,3117,PJM-YR-24-25,7,-0.760172,1.043428,-2.321037,PJM,-3.586514,-0.240411,2023-06-12 16:46:30.857


In [6]:
NodePrices_mapped = (
    NodePrices[
        ["AuctionKey", "PeriodCode", "NodeKey", "Cong24HMwh"]
    ]
    .merge(
        FTRAuctions[
            ["AuctionKey", "AuctionName", "AuctionEndDate"]
        ],
        on="AuctionKey",
        how="inner"
    )
    .merge(
        FTRAPeriods[
            ["AuctionKey", "PeriodCode", "StartDate", "EndDate"]
        ],
        on=["AuctionKey", "PeriodCode"],
        how="inner"
    )
)

NodePrices_mapped.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
0,3205,PJM-YR-26-27,3,-0.191220,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
1,3205,PJM-YR-26-27,4,4.888166,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
2,3205,PJM-YR-26-27,5,8.538418,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
3,3205,PJM-YR-26-27,6,19.659943,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
4,3205,PJM-YR-26-27,7,-3.970276,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000


In [7]:
df = NodePrices_mapped.copy()

df["AuctionEndDate"] = pd.to_datetime(df["AuctionEndDate"])
df["StartDate"] = pd.to_datetime(df["StartDate"])
df["EndDate"] = pd.to_datetime(df["EndDate"])

# Extract auction month from names like "FEB 2026 Auction"
auction_parts = df["AuctionName"].str.extract(
    r"([A-Z]{3})\s+(\d{4})"
)

df["AuctionMonth"] = pd.to_datetime(
    auction_parts[0] + " " + auction_parts[1],
    format="%b %Y",
    errors="coerce"
)

auction_price = df[
    # monthly periods only
    (df["StartDate"].dt.year == 2026)
    & (df["EndDate"].dt.year == 2026)
    & (df["StartDate"].dt.month == df["EndDate"].dt.month)

    # auction month = previous month of delivery month
    & (
        df["AuctionMonth"].dt.to_period("M")
        == (df["StartDate"].dt.to_period("M"))
    )
].copy()

auction_price = auction_price[
    [
        "AuctionKey",
        "PeriodCode",
        "NodeKey",
        "Cong24HMwh",
        "AuctionName",
        "AuctionEndDate",
        "StartDate",
        "EndDate",
    ]
]

auction_price.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
61890,3219,PJM-FEB-26,3,0.550610,FEB 2026 Auction,2026-01-21,2026-02-01,2026-02-28
61891,3219,PJM-FEB-26,4,1.782902,FEB 2026 Auction,2026-01-21,2026-02-01,2026-02-28
61892,3219,PJM-FEB-26,5,5.209687,FEB 2026 Auction,2026-01-21,2026-02-01,2026-02-28
61893,3219,PJM-FEB-26,6,12.696846,FEB 2026 Auction,2026-01-21,2026-02-01,2026-02-28
61894,3219,PJM-FEB-26,7,-8.465611,FEB 2026 Auction,2026-01-21,2026-02-01,2026-02-28


In [8]:
auction_price['AuctionName'].value_counts()

AuctionName
APR 2026 Auction    15538
MAY 2026 Auction    15538
MAR 2026 Auction    15292
FEB 2026 Auction    15291
Name: count, dtype: int64

## congestion spread

In [ ]:
congestion_spread = pd.read_csv('data/nodeDALMPH/negative_unilateral_extreme_pairs_congestion_spread.csv')

## reference price

In [ ]:
df = congestion_spread.copy()

df["date"] = pd.to_datetime(df["date"])

# ----------------------------------
# Historical months used for bootstrap
# ----------------------------------
bootstrap_df = df[
    (
        ((df["date"].dt.year == 2025) &
        (df["date"].dt.month.isin([4, 5, 6, 7])))
         # (df["date"].dt.month.isin([4, 5, 6])))
    )
    |
    (
        ((df["date"].dt.year == 2026) &
        (df["date"].dt.month == 4) &
        (df["date"].dt.day <= 15))
    )
].copy()

# ----------------------------------
# Double-weight May-2025
# ----------------------------------
may_2025 = bootstrap_df[
    (bootstrap_df["date"].dt.year == 2025)
    & (bootstrap_df["date"].dt.month == 5)
]

bootstrap_df = pd.concat(
    [bootstrap_df, may_2025],
    ignore_index=True
)

# apr_2026 = bootstrap_df[
#     (bootstrap_df["date"].dt.year == 2026)
#     & (bootstrap_df["date"].dt.month == 4)
# ]

# bootstrap_df = pd.concat(
#     [bootstrap_df, apr_2026],
#     ignore_index=True
# )

# ----------------------------------
# Forecast settings
# ----------------------------------
hours_in_may_2026 = 31 * 24
n_sims = 1000

results = []

# ----------------------------------
# Pair-by-pair bootstrap
# ----------------------------------
for pair_id, pair_df in bootstrap_df.groupby("pair"):

    spreads = pair_df["congestion_spread"].dropna().values
    
    # 10% lower tails
    q10 = np.quantile(spreads, 0.10)
    # q90 = np.quantile(spreads, 0.90)

    tail_data = spreads[(spreads <= q10)]

    # # duplicate tails once
    # spreads = np.concatenate([
    #     spreads,
    #     tail_data
    #     # np.repeat(tail_data, 3)
    # ])
    
    if len(spreads) < 100:
        continue

    # shape = (744 hours, 1000 simulations)
    simulations = np.random.choice(
        spreads,
        size=(hours_in_may_2026, n_sims),
        replace=True
    )

    # monthly average spread for each simulation
    sim_monthly_avg = simulations.mean(axis=0)

    reference_price = sim_monthly_avg.mean()

    results.append({
        "pair": pair_id,
        "reference_price": reference_price,
        "sim_std": sim_monthly_avg.std(),
        "sim_p05": np.quantile(sim_monthly_avg, 0.05),
        "sim_p50": np.quantile(sim_monthly_avg, 0.50),
        "sim_p95": np.quantile(sim_monthly_avg, 0.95),
        "n_hist_obs": len(spreads)
    })

reference_prices = pd.DataFrame(results)

reference_prices.head()